Importing all necessary libraries

In [24]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go

from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler

from statsmodels.distributions.empirical_distribution import ECDF

Importing features.parquet file

In [2]:
features_df = pd.read_parquet("data/processed/features.parquet")

In [3]:
features_df.head(5)

,flight_id,taxi_start,taxi_end,taxi_time_min,typecode,queue_size,avg_taxi_out_time_1h,n_intersections_passed,sum_seg_length_m,sum_p10_hist_seg_time,...,TO_runway_16,TO_runway_10,TO_runway_34,pushback,month,day_of_week,minute_of_day,minute_of_day_sin,minute_of_day_cos,temp_at_taxi_start
0,GSW6KP_4832,2024-12-01 04:44:56+00:00,2024-12-01 04:58:20+00:00,13.400000,A320,0,NaN,4,711.4,NaN,...,0,0,0,1,12,6,284,0.945519,0.325568,0.3
1,EDW200E_4558,2024-12-01 04:58:39+00:00,2024-12-01 05:07:58+00:00,9.316667,A320,0,13.400000,7,1190.3,NaN,...,0,0,0,1,12,6,298,0.963630,0.267238,0.3
2,EDW288G_3172,2024-12-01 05:04:09+00:00,2024-12-01 05:14:48+00:00,10.650000,A320,1,13.400000,8,1343.6,NaN,...,0,0,0,1,12,6,304,0.970296,0.241922,0.2
3,EDW120_4456,2024-12-01 05:18:52+00:00,2024-12-01 05:25:38+00:00,6.766667,A320,0,11.122222,4,740.8,NaN,...,0,0,0,1,12,6,318,0.983255,0.182236,0.2
4,EDW15T_4539,2024-12-01 05:31:07+00:00,2024-12-01 05:35:22+00:00,4.250000,A320,0,10.033333,7,1092.5,NaN,...,0,0,0,1,12,6,331,0.992005,0.126199,0.3


Sorting the DataFrame based on Time

In [4]:
features_df = (
    features_df
    .sort_values("taxi_start")
    .reset_index(drop=True)
)

Defining X (Features)

In [5]:
X = features_df[["typecode",
                 "queue_size",
                 "avg_taxi_out_time_1h",
                 "n_intersections_passed",
                 "sum_seg_length_m",
                 "sum_p10_hist_seg_time",
                 "sum_median_hist_seg_time",
                 "sum_p90_hist_seg_time",
                 "pushback",
                 "TO_runway_28",
                 "TO_runway_32",
                 "minute_of_day_sin",
                 "minute_of_day_cos",
                 "temp_at_taxi_start",
]]

Defining Y (Target Variable)

In [6]:
y = features_df[["taxi_time_min"]]

We perform a time-series-cross-validation. For this, we first have to define the folds, each consisting of training, validation, and test data


In [7]:
folds = [
    {
        "fold": 1,
        "val_start": "2025-03-01",
        "val_end": "2025-04-01",
        "test_start": "2025-04-01",
        "test_end": "2025-05-01",
    },
    {
        "fold": 2,
        "val_start": "2025-05-01",
        "val_end": "2025-06-01",
        "test_start": "2025-06-01",
        "test_end": "2025-07-01",
    },
    {
        "fold": 3,
        "val_start": "2025-07-01",
        "val_end": "2025-08-01",
        "test_start": "2025-08-01",
        "test_end": "2025-09-01",
    },
    {
        "fold": 4,
        "val_start": "2025-09-01",
        "val_end": "2025-10-01",
        "test_start": "2025-10-01",
        "test_end": "2025-11-01",
    },
    {
        "fold": 5,
        "val_start": "2025-11-01",
        "val_end": "2025-12-01",
        "test_start": "2025-12-01",
        "test_end": "2026-01-01",
    },
]

In a next step, we evaluate the model only based on training and validation data. This way, feature engineering can be performed

In [8]:
results = []

for fold_config in folds:

    fold = fold_config["fold"]
    val_start = fold_config["val_start"]
    val_end = fold_config["val_end"]

    print(f"\n========== FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_start

    val_mask = (
        (features_df["taxi_start"] >= val_start) &
        (features_df["taxi_start"] < val_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_val = features_df.loc[val_mask, X.columns]
    y_val = features_df.loc[val_mask, "taxi_time_min"]

    print(f"Training samples:   {len(X_train):,}")
    print(f"Validation samples: {len(X_val):,}")

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        early_stopping_rounds=50,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = root_mean_squared_error(y_val, y_pred)
    r_2 = r2_score(y_val, y_pred)

    results.append({
        "fold": fold,
        "mae": mae,
        "rmse": rmse,
        "r2": r_2
    })

results_df = pd.DataFrame(results)

mean_row = pd.DataFrame([{
    "fold": "Average",
    "mae": results_df["mae"].mean(),
    "rmse": results_df["rmse"].mean(),
    "r2": results_df["r2"].mean()
}])

results_df = pd.concat(
    [results_df, mean_row],
    ignore_index=True
)

results_df


========== FOLD 1 ==========
Training samples:   25,894
Validation samples: 9,283

========== FOLD 2 ==========
Training samples:   44,030
Validation samples: 9,583

========== FOLD 3 ==========
Training samples:   64,612
Validation samples: 10,977

========== FOLD 4 ==========
Training samples:   86,467
Validation samples: 10,030

========== FOLD 5 ==========
Training samples:   107,068
Validation samples: 9,253


,fold,mae,rmse,r2
0,1,2.753999,4.612901,0.396753
1,2,2.831126,4.759391,0.320190
2,3,2.729153,4.530442,0.358401
3,4,2.510222,4.099162,0.414641
4,5,2.777282,4.420163,0.527034
5,Average,2.720356,4.484412,0.403404



For each fold, hyperparameter tuning was performed separately using only the corresponding training and validation periods. By saving the best hyperparameters per fold seperately, we avoid data leakage. Otherwise, if we selected hyperparameters globally for all folds, earlier folds may have optimized parameters based on information that wasn't available at that time.

Twenty random hyperparameter combinations were sampled with `ParameterSampler` and evaluated on the validation set of the respective fold. The combination achieving the lowest validation RMSE was stored as the optimal hyperparameter configuration for that fold.

In [9]:
param_grid = {
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.5, 1, 2],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 2, 5],
    "learning_rate": [0.03, 0.05, 0.1],
}

param_samples = list(ParameterSampler(
    param_grid,
    n_iter=20,
    random_state=42
))

tuning_results = []
best_params_by_fold = {}

for fold_config in folds:

    fold = fold_config["fold"]
    val_start = fold_config["val_start"]
    val_end = fold_config["val_end"]

    print(f"\n========== TUNING FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_start

    val_mask = (
        (features_df["taxi_start"] >= val_start) &
        (features_df["taxi_start"] < val_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_val = features_df.loc[val_mask, X.columns]
    y_val = features_df.loc[val_mask, "taxi_time_min"]

    fold_results = []

    for i, params in enumerate(param_samples, start=1):

        print(f"Param set {i}/{len(param_samples)}")

        model = XGBRegressor(
            objective="reg:squarederror",
            enable_categorical=True,
            random_state=42,
            n_jobs=-1,
            n_estimators=3000,
            early_stopping_rounds=50,
            **params
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        result = {
            "fold": fold,
            "param_set": i,
            "val_start": val_start,
            "val_end": val_end,
            "rmse": rmse,
            "best_iteration": model.best_iteration,
            **params
        }

        tuning_results.append(result)
        fold_results.append(result)

    fold_results_df = pd.DataFrame(fold_results).sort_values("rmse")
    best_row = fold_results_df.iloc[0]

    best_params_by_fold[fold] = {
        param: best_row[param]
        for param in param_grid.keys()
    }

    print(f"\nBest params for Fold {fold}:")
    print(best_params_by_fold[fold])
    print(f"Best validation RMSE: {best_row['rmse']:.4f}")

tuning_results_df = pd.DataFrame(tuning_results)

tuning_results_df


========== TUNING FOLD 1 ==========
Param set 1/20
Param set 2/20
Param set 3/20
Param set 4/20
Param set 5/20
Param set 6/20
Param set 7/20
Param set 8/20
Param set 9/20
Param set 10/20
Param set 11/20
Param set 12/20
Param set 13/20
Param set 14/20
Param set 15/20
Param set 16/20
Param set 17/20
Param set 18/20
Param set 19/20
Param set 20/20

Best params for Fold 1:
{'max_depth': np.int64(6), 'min_child_weight': np.int64(3), 'subsample': np.float64(1.0), 'colsample_bytree': np.float64(1.0), 'gamma': np.float64(2.0), 'reg_alpha': np.float64(0.5), 'reg_lambda': np.int64(5), 'learning_rate': np.float64(0.1)}
Best validation RMSE: 4.5306

========== TUNING FOLD 2 ==========
Param set 1/20
Param set 2/20
Param set 3/20
Param set 4/20
Param set 5/20
Param set 6/20
Param set 7/20
Param set 8/20
Param set 9/20
Param set 10/20
Param set 11/20
Param set 12/20
Param set 13/20
Param set 14/20
Param set 15/20
Param set 16/20
Param set 17/20
Param set 18/20
Param set 19/20
Param set 20/20

Best 

,fold,param_set,val_start,val_end,rmse,best_iteration,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree
0,1,1,2025-03-01,2025-04-01,4.542117,403,0.8,5,0.0,5,4,0.05,2.0,0.8
1,1,2,2025-03-01,2025-04-01,4.547250,132,1.0,2,0.5,3,5,0.10,0.0,0.7
2,1,3,2025-03-01,2025-04-01,4.538180,285,1.0,5,0.1,3,5,0.05,0.5,0.8
3,1,4,2025-03-01,2025-04-01,4.563630,718,0.8,5,0.0,1,3,0.05,0.5,0.8
4,1,5,2025-03-01,2025-04-01,4.536593,235,0.8,5,0.5,1,6,0.05,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5,16,2025-11-01,2025-12-01,4.379802,821,0.8,2,0.1,3,4,0.10,0.0,0.7
96,5,17,2025-11-01,2025-12-01,4.366264,356,1.0,5,0.5,3,5,0.10,1.0,1.0
97,5,18,2025-11-01,2025-12-01,4.405533,1527,0.8,1,0.1,5,4,0.03,2.0,0.8
98,5,19,2025-11-01,2025-12-01,4.334710,1786,0.7,2,0.0,1,5,0.05,1.0,0.7


Now, we can apply these best hyperparameters on the final model. The folds now consist of training and test data, where training = training+validation from the previous model

In [14]:
final_results = []
prediction_results = []

for fold_config in folds:

    fold = fold_config["fold"]

    val_end = fold_config["val_end"]
    test_start = fold_config["test_start"]
    test_end = fold_config["test_end"]

    print(f"\n========== FINAL TEST FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_end

    test_mask = (
        (features_df["taxi_start"] >= test_start) &
        (features_df["taxi_start"] < test_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_test = features_df.loc[test_mask, X.columns]
    y_test = features_df.loc[test_mask, "taxi_time_min"]

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        **best_params_by_fold[fold]
    )

    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)

    prediction_results.append(pd.DataFrame({
        "fold": fold,
        "taxi_start": features_df.loc[test_mask, "taxi_start"].values,
        "y_true": y_test.values,
        "y_pred": y_pred
    }))

    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = root_mean_squared_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    val_row = results_df.loc[results_df["fold"] == fold].iloc[0]

    final_results.append({
        "fold": fold,
        "test_start": test_start,
        "test_end": test_end,

        "val_mae": val_row["mae"],
        "test_mae": test_mae,
        "mae_diff": test_mae - val_row["mae"],

        "val_rmse": val_row["rmse"],
        "test_rmse": test_rmse,
        "rmse_diff": test_rmse - val_row["rmse"],

        "val_r2": val_row["r2"],
        "test_r2": test_r2,
        "r2_diff": test_r2 - val_row["r2"],
    })

final_results_df = pd.DataFrame(final_results)

predictions_df = pd.concat(
    prediction_results,
    ignore_index=True
)

average_row = pd.DataFrame([{
    "fold": "Average",
    "test_start": "",
    "test_end": "",

    "val_mae": final_results_df["val_mae"].mean(),
    "test_mae": final_results_df["test_mae"].mean(),
    "mae_diff": final_results_df["mae_diff"].mean(),

    "val_rmse": final_results_df["val_rmse"].mean(),
    "test_rmse": final_results_df["test_rmse"].mean(),
    "rmse_diff": final_results_df["rmse_diff"].mean(),

    "val_r2": final_results_df["val_r2"].mean(),
    "test_r2": final_results_df["test_r2"].mean(),
    "r2_diff": final_results_df["r2_diff"].mean(),
}])

final_results_df = pd.concat(
    [final_results_df, average_row],
    ignore_index=True
)

final_results_df


========== FINAL TEST FOLD 1 ==========

========== FINAL TEST FOLD 2 ==========

========== FINAL TEST FOLD 3 ==========

========== FINAL TEST FOLD 4 ==========

========== FINAL TEST FOLD 5 ==========


,fold,test_start,test_end,val_mae,test_mae,mae_diff,val_rmse,test_rmse,rmse_diff,val_r2,test_r2,r2_diff
0,1,2025-04-01,2025-05-01,2.753999,3.039052,0.285053,4.612901,4.728490,0.115590,0.396753,0.278772,-0.117980
1,2,2025-06-01,2025-07-01,2.831126,2.970492,0.139365,4.759391,4.772670,0.013279,0.320190,0.315229,-0.004961
2,3,2025-08-01,2025-09-01,2.729153,2.658451,-0.070702,4.530442,4.263150,-0.267292,0.358401,0.403818,0.045418
3,4,2025-10-01,2025-11-01,2.510222,2.474591,-0.035631,4.099162,3.916526,-0.182636,0.414641,0.430975,0.016334
4,5,2025-12-01,2026-01-01,2.777282,2.717592,-0.059689,4.420163,4.371755,-0.048408,0.527034,0.501365,-0.025669
5,Average,,,2.720356,2.772036,0.051679,4.484412,4.410518,-0.073893,0.403404,0.386032,-0.017372


### Permutation Feature Importance on the Test Sets

Permutation Feature Importance (PFI) was calculated separately for each fold using the corresponding unseen test period. For every fold, the model was trained on the combined training and validation data using the fold-specific optimal hyperparameters. Feature importance was then determined by randomly permuting each feature 30 times and measuring the resulting increase in RMSE on the test set. The final importance values were obtained by averaging the fold-specific importances across all test folds.

In [11]:
# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

selected_features = [
    "typecode",
    "queue_size",
    "avg_taxi_out_time_1h",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "pushback",
    "TO_runway_28",
    "TO_runway_32",
    "minute_of_day_sin",
    "minute_of_day_cos",
    "temp_at_taxi_start",
]

feature_groups = {
    "typecode": ["typecode"],
    "queue_size": ["queue_size"],
    "avg_taxi_out_time_1h": ["avg_taxi_out_time_1h"],
    "n_intersections_passed": ["n_intersections_passed"],
    "sum_seg_length_m": ["sum_seg_length_m"],
    "sum_p10_hist_seg_time": ["sum_p10_hist_seg_time"],
    "sum_median_hist_seg_time": ["sum_median_hist_seg_time"],
    "sum_p90_hist_seg_time": ["sum_p90_hist_seg_time"],
    "pushback": ["pushback"],
    "TO_runway_28": ["TO_runway_28"],
    "TO_runway_32": ["TO_runway_32"],
    "minute_of_day": ["minute_of_day_sin", "minute_of_day_cos"],
    "temp_at_taxi_start": ["temp_at_taxi_start"],
}

feature_rename_map = {
    "typecode": "Aircraft Type Designator",
    "queue_size": "Queue Size",
    "avg_taxi_out_time_1h": "Average Taxi-Out Time Previous Hour [min]",
    "n_intersections_passed": "Number of Intersections Passed",
    "sum_seg_length_m": "Sum of all Segment Lengths [m]",
    "sum_p10_hist_seg_time": "Sum of all Segment Times 10th Percentile [s]",
    "sum_median_hist_seg_time": "Sum of all Segment Times Median [s]",
    "sum_p90_hist_seg_time": "Sum of all Segment Times 90th Percentile [s]",
    "pushback": "Pushback Required",
    "TO_runway_28": "Take-Off Runway 28",
    "TO_runway_32": "Take-Off Runway 32",
    "minute_of_day": "Minute of Day",
    "temp_at_taxi_start": "Air Temperature [°C]",
}

feature_group_map = {
    "minute_of_day": "Temporal Feature",
    "temp_at_taxi_start": "Meteorological Feature",
}

for feature in feature_groups:
    if feature not in feature_group_map:
        feature_group_map[feature] = "Operational Feature"

group_colour_map = {
    "Operational Feature": "royalblue",
    "Temporal Feature": "darkorange",
    "Meteorological Feature": "darkgreen",
}

n_repeats = 30
rng = np.random.default_rng(42)

perm_results = []

for fold_config in folds:

    fold = fold_config["fold"]

    val_end = fold_config["val_end"]
    test_start = fold_config["test_start"]
    test_end = fold_config["test_end"]

    print(f"\n========== TEST PFI FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_end

    test_mask = (
        (features_df["taxi_start"] >= test_start) &
        (features_df["taxi_start"] < test_end)
    )

    X_train = features_df.loc[train_mask, selected_features].copy()
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_test = features_df.loc[test_mask, selected_features].copy()
    y_test = features_df.loc[test_mask, "taxi_time_min"]

    X_train["typecode"] = X_train["typecode"].astype("category")
    X_test["typecode"] = X_test["typecode"].astype("category")

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        **best_params_by_fold[fold]
    )

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    baseline_pred = model.predict(X_test)
    baseline_rmse = root_mean_squared_error(y_test, baseline_pred)

    for feature_name, cols in feature_groups.items():

        importances = []

        for repeat in range(n_repeats):
            X_test_perm = X_test.copy()

            shuffled_idx = rng.permutation(len(X_test_perm))

            for col in cols:
                X_test_perm[col] = X_test_perm[col].to_numpy()[shuffled_idx]

            X_test_perm["typecode"] = X_test_perm["typecode"].astype("category")

            perm_pred = model.predict(X_test_perm)
            perm_rmse = root_mean_squared_error(y_test, perm_pred)

            importances.append(perm_rmse - baseline_rmse)

        perm_results.append({
            "fold": fold,
            "feature": feature_name,
            "importance_mean": np.mean(importances),
            "importance_std": np.std(importances),
            "baseline_rmse": baseline_rmse
        })

perm_importance_df = pd.DataFrame(perm_results)

perm_summary_df = (
    perm_importance_df
    .groupby("feature")
    .agg(
        mean_importance=("importance_mean", "mean"),
        std_importance=("importance_mean", "std"),
        mean_perm_std=("importance_std", "mean")
    )
    .sort_values("mean_importance", ascending=False)
    .reset_index()
)

plot_df = perm_summary_df.copy()
plot_df["label"] = plot_df["feature"].map(feature_rename_map).fillna(plot_df["feature"])
plot_df["group"] = plot_df["feature"].map(feature_group_map).fillna("Operational Feature")

plot_df = plot_df.sort_values("mean_importance", ascending=True)

fig = go.Figure()

category_order = plot_df["label"].tolist()

for group_name, sub in plot_df.groupby("group"):
    fig.add_trace(
        go.Bar(
            x=sub["mean_importance"],
            y=sub["label"],
            orientation="h",
            error_x=dict(
                type="data",
                array=sub["std_importance"],
                visible=True,
                thickness=3,
                width=10
            ),
            marker=dict(color=group_colour_map[group_name]),
            name=group_name,
            showlegend=True
        )
    )

fig.update_layout(
    height=900,
    width=1500,
    margin=dict(t=50, b=70, l=430, r=50),

    font=dict(
        family=font_family,
        color=font_colour
    ),

    title=None,

    xaxis=dict(
        title="Mean Increase in RMSE [min]",
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    yaxis=dict(
        title="Features",
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size),
        categoryorder="array",
        categoryarray=category_order
    ),

    legend=dict(
        title="Feature Groups",
        font=dict(size=legend_font_size),
        title_font=dict(size=legend_title_size),
        x=0.99,
        y=0.01,
        xanchor="right",
        yanchor="bottom"
    ),
)

fig.show()

fig.write_image("Permutation_Feature_Importance_RMSE_Test.pdf")

perm_summary_df


========== TEST PFI FOLD 1 ==========

========== TEST PFI FOLD 2 ==========

========== TEST PFI FOLD 3 ==========

========== TEST PFI FOLD 4 ==========

========== TEST PFI FOLD 5 ==========


,feature,mean_importance,std_importance,mean_perm_std
0,sum_p10_hist_seg_time,1.264675,0.690052,0.031008
1,sum_p90_hist_seg_time,1.117383,0.465102,0.024038
2,sum_median_hist_seg_time,0.984857,0.265989,0.022147
3,n_intersections_passed,0.422586,0.079998,0.012314
4,sum_seg_length_m,0.361896,0.083477,0.013021
5,queue_size,0.343899,0.049587,0.013820
6,typecode,0.335823,0.042754,0.017537
7,minute_of_day,0.245017,0.138095,0.016400
8,TO_runway_28,0.068429,0.049871,0.006149
9,avg_taxi_out_time_1h,0.059120,0.034127,0.008078


In a next step, we can create plots based on these results

## Actual vs Predicted Taxi-Out Times for ALL test sets

In [20]:
# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# Ensure y_true and y_pred are 1-D sequences
y_true = predictions_df["y_true"]
y_pred = predictions_df["y_pred"]

# Define axis limits
x_min = 0
x_max = 60

y_min = 0
y_max = 60

# Create the ideal prediction line (y = x)
ideal_x = [x_min, x_max]
ideal_y = [x_min, x_max]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=y_true,
        y=y_pred,
        mode="markers",
        name="Predicted vs Actual",
        marker=dict(
            color="navy",
            size=7,
            opacity=0.35
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=ideal_x,
        y=ideal_y,
        mode="lines",
        name="Ideal Line",
        line=dict(
            color="red",
            width=4,
            dash="dash"
        )
    )
)

fig.update_layout(
    height=700,
    width=1500,
    margin=dict(t=50, b=100, l=100, r=50),

    title=None,

    font=dict(
        family=font_family,
        color=font_colour
    ),

    xaxis=dict(
    title="Actual Taxi-Out Time [min]",
    range=[0, 60],
    fixedrange=True,
    title_font=dict(size=axis_title_size),
    tickfont=dict(size=axis_tick_size)
),

    yaxis=dict(
    title="Predicted Taxi-Out Time [min]",
    range=[0, 60],
    fixedrange=True,
    title_font=dict(size=axis_title_size),
    tickfont=dict(size=axis_tick_size)
),

    legend=dict(
        title="Legend",
        font=dict(size=legend_font_size),
        title_font=dict(size=legend_title_size),
        x=0.99,
        y=0.01,
        xanchor="right",
        yanchor="bottom"
    )
)

fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

fig.write_image("Predicted_vs_Actual_Taxi_Out_Time.pdf")

## Residuals vs Predicted Taxi-Out Times

In [23]:
# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# Ensure y_true and y_pred are 1-D
y_true = predictions_df["y_true"]
y_pred = predictions_df["y_pred"]

# Compute residuals
residuals = y_true - y_pred

# Axis limits
x_min = 0
x_max = 60

y_min = -40
y_max = 60

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=y_pred,
        y=residuals,
        mode="markers",
        name="Residuals",
        marker=dict(
            color="navy",
            size=7,
            opacity=0.35
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=[x_min, x_max],
        y=[0, 0],
        mode="lines",
        name="Zero Error Line",
        line=dict(
            color="red",
            width=4,
            dash="dash"
        )
    )
)

fig.update_layout(
    height=700,
    width=1500,
    margin=dict(t=50, b=100, l=100, r=50),

    title=None,

    font=dict(
        family=font_family,
        color=font_colour
    ),

    xaxis=dict(
        title="Predicted Taxi-Out Time [min]",
        range=[x_min, x_max],
        fixedrange=True,
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    yaxis=dict(
        title="Residual (Actual - Predicted Taxi-Out Time) [min]",
        range=[y_min, y_max],
        fixedrange=True,
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    legend=dict(
        title="Legend",
        font=dict(size=legend_font_size),
        title_font=dict(size=legend_title_size),
        x=0.99,
        y=0.01,
        xanchor="right",
        yanchor="bottom"
    )
)

fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

fig.show()

fig.write_image("Residuals_vs_Predicted_Taxi_Out_Time.pdf")

ECDF

In [27]:
# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# Compute residuals
errors = y_true - y_pred

# ECDF
ecdf = ECDF(errors)

# Percentiles
q10, q50, q90 = np.percentile(errors, [10, 50, 90])

# Create figure
fig = go.Figure()

# ECDF curve
fig.add_trace(
    go.Scatter(
        x=ecdf.x,
        y=ecdf.y,
        mode="lines",
        name="ECDF",
        line=dict(
            width=4,
            color="navy"
        )
    )
)

# 10th percentile
fig.add_trace(
    go.Scatter(
        x=[q10, q10],
        y=[0, 1],
        mode="lines",
        name=f"10th Percentile ({q10:.2f} min)",
        line=dict(
            width=4,
            dash="dash",
            color="darkorange"
        ),
        showlegend=True
    )
)

# Median
fig.add_trace(
    go.Scatter(
        x=[q50, q50],
        y=[0, 1],
        mode="lines",
        name=f"Median ({q50:.2f} min)",
        line=dict(
            width=4,
            dash="dash",
            color="red"
        ),
        showlegend=True
    )
)

# 90th percentile
fig.add_trace(
    go.Scatter(
        x=[q90, q90],
        y=[0, 1],
        mode="lines",
        name=f"90th Percentile ({q90:.2f} min)",
        line=dict(
            width=4,
            dash="dash",
            color="darkgreen"
        ),
        showlegend=True
    )
)

# Layout styling
fig.update_layout(
    height=700,
    width=1500,
    margin=dict(t=50, b=100, l=100, r=50),

    title=None,

    font=dict(
        family=font_family,
        color=font_colour
    ),

    xaxis=dict(
        title="Residual (Actual - Predicted Taxi-Out Time) [min]",
        range=[-30, 40],
        fixedrange=True,
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    yaxis=dict(
        title="Cumulative Probability",
        range=[0, 1.02],
        fixedrange=True,
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    legend=dict(
        title="Legend",
        font=dict(size=legend_font_size),
        title_font=dict(size=legend_title_size),
        x=0.99,
        y=0.01,
        xanchor="right",
        yanchor="bottom"
    )
)

# Grid
fig.update_xaxes(
    showgrid=True,
    gridwidth=0.5,
    gridcolor="lightgrey"
)

fig.update_yaxes(
    showgrid=True,
    gridwidth=0.5,
    gridcolor="lightgrey"
)

fig.show()

fig.write_image("ECDF_Residuals.pdf")